# Standby Duty Schedule

note down all sources!

## Current Situation

- always 90 standby drives (baseline model)

In [ ]:
# next steps
# o TODOs below
# o add to word in parallel

## Exploratory Data Analysis

| Column | Datatype | Description |
| --- | --- | --- |
| date | string | yyyy-mm-dd |
| n_sick | int | amount of sick drivers |
| calls | float | emergency calls |
| n_duty | int | amount of **on-duty** drivers |
| n_sby | int | amount of available **standby** drivers |
| sby_need | float | amount of activated **standby** drivers |
| dafted | float | drafted **off-duty drivers** if standby drivers are not enough |

In [ ]:
# imports 
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

%matplotlib ipympl

# import dataset
df = pd.read_csv("sickness_table.csv")

# delete indexing variable of dataset
if "Unnamed: 0" in df.columns:
    del df["Unnamed: 0"]

### Evaluation of Data Quality

In [ ]:
# check if there are any empty cells
if not df.isnull().any().any():
    print("There are no empty cells in the dataset.")

# check if all floats are actually whole numbers (x.0) and parse to int
float_cols = ["calls", "sby_need", "dafted"]
parse_floats_to_ints = True
for col in float_cols:
    if not all(df[col].apply(float.is_integer)):
        print(f"Column {col} contains non-integer floats.")
        parse_floats_to_ints = False
if parse_floats_to_ints:
    print("All float columns contain only whole numbers. Parsing to int.")
    df[float_cols] = df[float_cols].astype(int)

# check if all dates follow yyyy-mm-dd schema
if all(df["date"].str.match(r"\d{4}-\d{2}-\d{2}")):
    print("All dates follow the yyyy-mm-dd schema.")
# check if data for all days is available
start_date = df["date"].min()
end_date = df["date"].max()
if not any(pd.date_range(start=start_date, end=end_date).difference(pd.to_datetime(df["date"]))):
    print("Data is available for all days.")

In [ ]:
# create profile report
profile = ProfileReport(df, title="Sickness Table Report")
profile.to_notebook_iframe()

### Feature generation

then: - visualize important connections/correlation (easily understandable, also show quality of data)

In [ ]:
# create additional features: d, m , y and delete original date column
df[["year", "month", "day"]] = df["date"].str.split("-", expand=True).astype(int)

# create additional feature: n_work (actually working drivers)
df["n_work"] = df["n_duty"] - df["n_sick"] + df["sby_need"]

# percentage of sick drivers since n_duty changes
df["perc_sick"] = df["n_sick"] / df["n_duty"]

# TODO: don't extrapolate trend if not used in model (then only needed for EDA)
# get trend, seasonal and residual components of calls (avoiding NaNs w/ extrapolation)
result_calls = seasonal_decompose(df["calls"], model="additive", period=365, extrapolate_trend="freq")
df["calls_trend"] = result_calls.trend
df["calls_seas"] = result_calls.seasonal
df["calls_resid"] = result_calls.resid

# get trend, seasonal and residual components of perc_sick (avoiding NaNs w/ extrapolation)
result_perc_sick = seasonal_decompose(df["perc_sick"], model="additive", period=365, extrapolate_trend="freq")
df["perc_sick_trend"] = result_perc_sick.trend
df["perc_sick_seas"] = result_perc_sick.seasonal
df["perc_sick_resid"] = result_perc_sick.resid

In [ ]:
# plotting decompositions to show trends, seasonality, and residuals
result_perc_sick.plot()
result_calls.plot()
plt.show()

In [ ]:
# general plotting
plt.close("all")
df_plot = df.drop(columns=["n_sby", "month", "day", "date"])  # w/o uninformative columns
fig, axs = plt.subplots(len(df_plot.columns) + 1, 1, figsize=(10, 2 * len(df_plot.columns)), sharex=True)
for i, ax in enumerate(axs):
    if i < len(axs) - 1:
        ax.plot(df_plot.iloc[:, i])
        ax.set_title(df_plot.columns[i])
    else:
        # evaluation metric: percentage of sby_need compared to n_sby
        ax.plot(100 * df["sby_need"] / df["n_sby"])
        ax.hlines(100, xmin=df.index.min(), xmax=df.index.max(), colors="r", linestyles="dashed")
        ax.set_title("Percentage of sby_need compared to n_sby")
fig.tight_layout()

In [ ]:
# plot relationship between n_work and calls for each n_duty
# better than plot in sns pairplot since noise got reduced
n_duties = df["n_duty"].unique()
fig, ax = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
for n_duty in n_duties:
    ax.scatter(
        df["n_work"].where(df["n_duty"] == n_duty),
        df["calls"].where(df["n_duty"] == n_duty),
        marker="x",
        label=f"n_duty: {n_duty}",
    )
ax.scatter(
    df["n_work"].where(df["sby_need"] > 0),
    df["calls"].where(df["sby_need"] > 0),
    alpha=0.4,
    marker=".",
    color="red",
    label="sby_need > 0",
)
ax.legend()
ax.set_xlabel("n_work")
ax.set_ylabel("calls")
ax.grid()
fig.tight_layout()

In [ ]:
# calculate correlation between n_work and calls w/o sby_need for each n_duty
print("corr between n_work and calls (only if standby drivers were needed)")
for n_duty in n_duties:
    indices = df.index[(df["sby_need"] > 0) & (df["n_duty"] == n_duty)]
    n_work = df.loc[indices, "n_work"]
    calls = df.loc[indices, "calls"]
    r = np.corrcoef(n_work, calls)
    print(f"\tFor n_duty={n_duty} the corr is {r[1, 0]}")
print("\nremaining n_work data points are solely deviating from n_duty because of n_sick")

In [ ]:
# plot correlation matrix
# except for n_sby since it's always 90 and other uninformative columns
df.drop(columns=["n_sby", "date", "n_sick", "calls", "perc_sick"]).corr().style.background_gradient(
    cmap="coolwarm", axis=None, vmin=-1, vmax=1
)

In [ ]:
# Findings
# general:
#   - for n_work above n_duty-n_sick correlation between calls and n_work is very strong
#     (for lower n_work there is a lot of noise due to n_sick, but upper limit follows correlation)
#   - n_sick and sby_need/dafted are not related
#   - sby_needed is the same as drafted just w/ offset of n_sby
#   - n_duty got elevated every year
# correlations:
#   - the more n_duty, the more n_sick (obvious)
#   - the more n_duty, the more n_work (obvious)
#   - the later, the more calls
#   - the more calls, the more sby_needed w/ offset because of n_duty (obvious)
#   - most calls in summer
# - weak:
#   - the more n_sick, the less n_duty (obvious)
#   - the more calls, the more n_sick (very weak)

In [ ]:
# plot correlations (except for uninformative columns)
sns.pairplot(df.drop(columns=["n_sby", "n_sick", "n_duty", "dafted", "date"]), height=1.5)

## Model

### Characteristics

- autocorrelated data
- time series data

### Tasks

- feature selection (w/ which technic?) (avoid data leakage of data that is not available when planning)
- choose model (only go for one approach, show why others were not pursued)
- avoid overfitting (how?)

### Objectives

- predict on a daily basis the amount of standby drivers efficiently (maximize rate of called in standby drivers)
- minimize days w/ too little drivers (&rarr; dafted drivers needed) (might result in very little days w/ way to many missing drivers, look out for that)
- -> exploitation rate of n_sby for days w/ sby_need/n_sby < 1 and amount of days w/ sby_need/n_sby > 1 

### Restrictions

- plan will be created on the 15th for the following month -> last half of month should not be included in training data

### Further Requirements

- discuss feature importance to increase trust in model (not applicable)
- make predictions as interpretable as possible
- detailed failure analysis to asses situations for which model is not suited
    - plot error in histogram (should be gauss if accumulation somewhere inspect those samples and look for commonalities)
- look up script for only using 90th percentile (for more robustness)

In [ ]:
# overall structure (_pred for predicted features)
# n_sby_pred = n_work_pred + n_sick_pred - n_duty (transformed from calculation of n_work above)
#   n_work_pred predicted by linear regression of calls_pred
#       calls_pred is predicted by time series prediction
#   n_sick_pred is predicted by time series prediction

# TODO:
# o preparation:
#   o Kaggle: extraction of day of week, month, etc. (day/mon/... can be modelled w/ sin/cos (to prevent jump of -11))
#   o try _seas w/ (P)ACF (just to see what happens)
#   o for what is validation dataset used? (just for validating hyperparameters?)
#   o look up kernel smoothing and exponential smoothing (alternatives to moving average)
#   o FFT (w/ plt function) of calls and n_sick
#   o check for correlation of weekdays (1-7) (one-hot w/ "wday_1", ...) / holidays (binary distinction)
#   o month/days/years should be transformed using trigonometric functions (look that up) or one-hot encoding
#   o other (proper) ways of feature selection (wrapper methods: forward selection, backward elimination 
#     (lecture 7, how does improvement through feature get evaluated))
#   o look up state space models (kalman filter, what does only applicable to linear systems mean?) (lecture 6)
# continue lecture 7, p. 74 (forecasting intervals, research is needed)

# o autocorrelation? (continue in GG at time series analysis)
# o research:
#   SARIMA, ARIMA, XGBoost
#   LSTM, RNN, Temporal CNN for time series data (but need many data for training)
#   (extended) Kalman-Filter w/ physical models,
#   or no time series data at all but random forest
#   GeeksForGeeks: list of algos
# o save dataset which is used for training (maybe train in new notebook)
# o develop model
#   o predict calls (for that only date and previous calls (trend, seasonal, residuals) are necessary (everything else is not related))
#   o prediction needs to be on a daily basis
#     (e.g. w/ lib: from statsforecast import StatsForecast)
#   o relevant columns: calls_trend, calls_seas, calls_resid, year, month, day, perc_sick_trend, perc_sick_seas, perc_sick_resid
#     (probably only calls and perc_sick needed (or are trend and seas helpful as well?))
# o possibilities to include days w/ too little n_sby and exploitation rate in penalty term?
#   (would make a retraining of submodels during training of overall model necessary)
#   (would work w/ simple model for calls_pred and n_sick_pred) (or just tweaking quantiles/hyperparameters of training)
# o point estimator (e.g. w/ mu and sigma (according distribution which is similar to distribution of feature itself (here Gaus I think))
#   can be trained on higher level? -> then with goals (exploitation and little days >100%)
#   use quantil in cost function? (p. 80)

# o avoid overfitting through trainings and testdata
# o use hyperparameter (determine w/ cross validation, brute force?, w 3rd dataset)
#   cross validation (usually w/ small datasets) w/ time series: 
#   make sure that model sees all seasonalities in one training split (test also needs to be for all seasons)
#   fixed origin: 1. train, 2. test; 1.+2. train, 3.test; ... (1. = time interval)
#   rolling window: 1. train, 2. test; 2. train, 3.test; ... (test always smaller than train)
# o penalty terms for reducing complexity here applicable?
# o check for structure in the residuals of the model (there should be none) (plot residual over time, in histogram, in probplot, 
#   in correlogram (lags should not be correlated at all (only 1.)))
# o evaluation of model!
#   o metrics in lection 7 p. 64

# o make interpretable
#   o linear models/GLM (generalized linear models) show influence of features w/ coefficients
#   o (decision trees are completely interpretable but here not applicable)
#   o are there surrogate-models for the time series models which can approximate individual predictions (and therefore) explain them?

# o for restriction of needing time plan already on the 15th: just predict always 1.5 months

In [ ]:
plot_acf(df["calls"], lags=50)
plt.show()

In [ ]:
plot_acf(df["perc_sick"], lags=50)
plt.show()

In [ ]:
plot_pacf(df["calls"], lags=50)
plt.show()

In [ ]:
plot_pacf(df["perc_sick"], lags=50)
plt.show()